In [1]:
import os
import re
import csv
from collections import defaultdict
from statistics import mean

def parse_log_file_last_metrics(path):
    """从 log 文件末尾提取最后一次 eval 的指标（支持科学计数法）"""
    eval_metrics = {}
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()

    i = len(lines) - 1
    while i >= 0:
        if lines[i].strip().startswith("start to eval"):
            # hit20
            if i + 1 < len(lines) and lines[i+1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.+-eE]+)", lines[i+1])
                if m:
                    eval_metrics["hit20"] = float(m.group(1))
            # hit50
            if i + 2 < len(lines) and lines[i+2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.+-eE]+)", lines[i+2])
                if m:
                    eval_metrics["hit50"] = float(m.group(1))
            # hit100
            if i + 3 < len(lines) and lines[i+3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.+-eE]+)", lines[i+3])
                if m:
                    eval_metrics["hit100"] = float(m.group(1))
            # roc_auc 等
            if i + 4 < len(lines) and lines[i+4].startswith("roc_auc"):
                nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", lines[i+4])
                if len(nums) >= 5:
                    fields = ['roc_auc', 'pr_auc', 'f1', 'mrr_pess', 'mrr_opt']
                    result_dict = dict(zip(fields, map(float, nums[-5:])))
                    eval_metrics.update(result_dict)
            if eval_metrics:
                break
        i -= 1
    return eval_metrics


# ==== 参数 ====
dataset = "ogbl_citation2"
ratio = 0.02
cs = [1, 2]
convergences = [0.9, 1.0]
pos_ratios = [0.8, 1.0, 1.2]
seeds = [1, 2, 3, 4, 5]

log_base_dir = "."
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

ALL_METRICS = ['hit20', 'hit50', 'hit100', 'roc_auc', 'pr_auc', 'f1', 'mrr_pess', 'mrr_opt']

metric_bucket = defaultdict(list)
metric_keys = set()

# ==== 解析日志 ====
for c in cs:
    for conv in convergences:
        for pr in pos_ratios:
            for seed in seeds:
                fname = f"c{c}-conv{conv}-pos{pr}-s{seed}-r{ratio}.log"
                log_path = os.path.join(log_base_dir, fname)
                key = (c, conv, pr, seed)

                if not os.path.isfile(log_path):
                    print(f"[WARN] 缺失: {log_path}")
                    metrics = {k: 0.0 for k in ALL_METRICS}
                    metric_bucket[key].append(metrics)
                    metric_keys.update(metrics.keys())
                    continue

                metrics = parse_log_file_last_metrics(log_path)
                if not metrics:
                    print(f"[WARN] 无指标: {log_path}")
                    metrics = {k: 0.0 for k in ALL_METRICS}
                metric_bucket[key].append(metrics)
                metric_keys.update(metrics.keys())

# ==== 写入 CSV ====
csv_path = os.path.join(output_dir, f"{dataset}_grid_result.csv")
metric_keys = sorted(metric_keys)
header = ["c", "convergence", "pos_ratio", "seed"] + metric_keys

with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    for key, metrics_list in metric_bucket.items():
        c, conv, pr, seed = key
        row = {"c": c, "convergence": conv, "pos_ratio": pr, "seed": seed}
        for k in metric_keys:
            vals = [m[k] for m in metrics_list if k in m]
            row[k] = f"{round(round(mean(vals), 8) * 100, 4)}" if vals else "0.0000"
        writer.writerow(row)

print(f"[INFO] 写入完毕: {csv_path}")


[INFO] 写入完毕: ./results/ogbl_citation2_grid_result.csv


In [2]:
import pandas as pd
import numpy as np
from tabulate import tabulate

# === 读取数据 ===
csv_path = "./results/ogbl_citation2_grid_result.csv"
df = pd.read_csv(csv_path)
df["mrr_pess"] = df["mrr_pess"].astype(float)

# === 按 (c, convergence, pos_ratio) 分组求均值和方差 ===
summary = (
    df.groupby(["c", "convergence", "pos_ratio"])["mrr_pess"].agg(["mean", "std"]).reset_index()
)


# === 展示结果表 ===
print("========= MRR 统计结果 (mean ± std) =========")
table = summary[["c", "convergence", "pos_ratio", "mean", "std"]]
print(tabulate(table, headers="keys", tablefmt="github", floatfmt=".4f"))


========= MRR 统计结果 (mean ± std) =========
|    |      c |   convergence |   pos_ratio |   mean |    std |
|----|--------|---------------|-------------|--------|--------|
|  0 | 1.0000 |        0.9000 |      0.8000 | 0.0858 | 0.0081 |
|  1 | 1.0000 |        0.9000 |      1.0000 | 0.0832 | 0.0063 |
|  2 | 1.0000 |        0.9000 |      1.2000 | 0.0890 | 0.0075 |
|  3 | 1.0000 |        1.0000 |      0.8000 | 0.1078 | 0.0212 |
|  4 | 1.0000 |        1.0000 |      1.0000 | 0.1058 | 0.0172 |
|  5 | 1.0000 |        1.0000 |      1.2000 | 0.1137 | 0.0138 |
|  6 | 2.0000 |        0.9000 |      0.8000 | 0.0916 | 0.0155 |
|  7 | 2.0000 |        0.9000 |      1.0000 | 0.0900 | 0.0103 |
|  8 | 2.0000 |        0.9000 |      1.2000 | 0.0856 | 0.0095 |
|  9 | 2.0000 |        1.0000 |      0.8000 | 0.1012 | 0.0185 |
| 10 | 2.0000 |        1.0000 |      1.0000 | 0.0954 | 0.0263 |
| 11 | 2.0000 |        1.0000 |      1.2000 | 0.1016 | 0.0201 |
